# Parte II — Markov y Monte Carlo

**H3:** 20/10 · **H4:** 27/10 · **H5 preliminar:** 03/11 ·
**entrega:** 06/11, 18:00 · **defensa:** 09/11, 9:00–13:00

Este cuaderno evalúa, sin reoptimizar, las dos políticas creadas en la Parte I.
H5 revisa la reproducción, el informe y la presentación ya preparados sobre el
estado congelado en H4: no se agrega un método ni experimento nuevo después del
segundo parcial.

## 1. Contrato previo (H3)

Antes de ejecutar la campaña, documente:

1. estados y orden fijo (Normal, Pico, Disrupcion);
2. condición inicial \(Z_1=\text{Disrupción}\);
3. horizonte de 26 semanas y orden de eventos;
4. distribuciones, dependencias e independencia condicional;
5. políticas nominal y resguardo, ambas fijas;
6. estimandos, convención resguardo - nominal, semilla 20260816;
7. piloto de 200, campaña final de 1200 e invariantes.

H3 no contiene resultados: es un diseño previo.

- Estados en orden fijo: **Normal, Pico, Disrupcion**. La semana 1 comienza en Disrupcion, último estado observado.
- En cada semana y zona, la demanda se sortea de una Poisson con media `nominal_demand × demand_multiplier`. Condicionalmente al estado, las cinco demandas son independientes.
- En cada semana y ruta, la disponibilidad se sortea como Bernoulli con probabilidad `route_availability`. Condicionalmente al estado, los sorteos de rutas son independientes entre sí y de las demandas. Si opera, su capacidad real es `capacity × capacity_multiplier`; si falla, es cero.
- Se intenta despachar el `planned_flow` de cada plan, limitado por la capacidad real. No se agrega una decisión nueva ni se reoptimiza. El inventario inicial o remanente se suma a lo recibido; se atiende demanda, se registra faltante perdido, se conserva hasta `storage_capacity` y se descarta el exceso.
- Cada trayectoria dura 26 semanas. Semilla maestra **20260816**; las primeras **200** de **1200** réplicas forman el piloto. El diseño se mantiene y las 1200 entran en la estimación final.
- Una réplica genera **un único futuro** y lo entrega a ambas políticas. Todas las diferencias se calculan como **resguardo menos nominal**.

**Métricas por trayectoria:** costo total; nivel de servicio = unidades atendidas / demanda total; unidades faltantes; indicador de algún faltante; fracción de semanas con faltante; inventario medio al cierre (suma de las cinco zonas promediada sobre las 26 semanas); inventario final; descarte y unidades despachadas. Si una prueba artificial tiene demanda total cero, se define servicio igual a 1.

El costo suma transporte ejecutado, penalización por faltante, conservación de inventario de cierre y descarte. La comparación principal usa la media, error estándar e intervalo t bilateral del 95 % de la diferencia emparejada de costo y servicio.

## 2. Entradas y archivo puente

La ausencia o alteración manual del puente debe detener la ejecución.

In [8]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import t as student_t

# El notebook se puede ejecutar desde Parte2 o desde la raíz del proyecto.
DATA_CANDIDATES = [Path('../Datos'), Path('Datos')]
DATA = next((path.resolve() for path in DATA_CANDIDATES if path.is_dir()), None)
assert DATA is not None, 'No se encontró Datos/. Ejecutá desde Parte2 o desde la raíz del proyecto.'
OUTPUTS = DATA.parent / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

nominal_path = OUTPUTS / 'part1_solution.csv'
buffer_path = OUTPUTS / 'part1_solution_buffer.csv'
assert nominal_path.is_file() and buffer_path.is_file(), 'Faltan los CSV puente: ejecutá primero la Parte I.'

# Hashes congelados de los planes aprobados. Si se regenera la Parte I, revisar y actualizar estos valores.
EXPECTED_BRIDGE_SHA256 = {
    'part1_solution.csv': '79c91cb78ceb0ebed1c98c12bade96d14b2abecf5ecde2b2640df0592476b5ab',
    'part1_solution_buffer.csv': 'ce6330cb655529ef58c9b292da49789d2c1598e77f589f794f1b55b445192e23',
}

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

for path in (nominal_path, buffer_path):
    actual_hash = sha256(path)
    expected_hash = EXPECTED_BRIDGE_SHA256[path.name]
    assert actual_hash == expected_hash, (
        f'{path.name} no coincide con el puente congelado. '
        f'Esperado: {expected_hash}; actual: {actual_hash}. '
        'Revisá la Parte I y actualizá el hash solo si el cambio fue aprobado.'
    )
    print(f'SHA-256 {path.name}: {actual_hash}')

nominal_plan = pd.read_csv(nominal_path)
buffer_plan = pd.read_csv(buffer_path)
observations = pd.read_csv(DATA / 'markov_observations.csv')
state_parameters = pd.read_csv(DATA / 'markov_state_parameters.csv')
destinations = pd.read_csv(DATA / 'destinations.csv')
routes = pd.read_csv(DATA / 'routes.csv')

required_plan_columns = {
    'route_id', 'origin', 'destination', 'planned_flow', 'capacity', 'unit_cost', 'policy'
}
for name, plan in [('nominal', nominal_plan), ('resguardo', buffer_plan)]:
    assert required_plan_columns <= set(plan.columns), f'Columnas faltantes en plan {name}.'
    assert plan['route_id'].is_unique and len(plan) == len(routes), f'Rutas incompletas o duplicadas en plan {name}.'
    assert set(plan['route_id']) == set(routes['route_id']), f'Las rutas de {name} no coinciden con routes.csv.'
    assert plan['planned_flow'].notna().all() and (plan['planned_flow'] >= 0).all(), f'Flujos inválidos en plan {name}.'
    check = plan.merge(routes, on='route_id', suffixes=('_plan', '_data'), validate='one_to_one')
    for column in ('origin', 'destination'):
        assert (check[f'{column}_plan'] == check[f'{column}_data']).all(), f'{column} no coincide en plan {name}.'
    for column in ('capacity', 'unit_cost'):
        assert np.allclose(check[f'{column}_plan'], check[f'{column}_data']), f'{column} no coincide en plan {name}.'

assert observations['period'].is_unique, 'period debe ser único en markov_observations.csv.'
assert set(observations['state']) <= {'Normal', 'Pico', 'Disrupcion'}, 'Hay estados desconocidos en las observaciones.'
required_state_columns = {'state', 'demand_multiplier', 'capacity_multiplier', 'route_availability'}
assert required_state_columns <= set(state_parameters.columns), 'Faltan columnas requeridas en markov_state_parameters.csv.'
assert set(state_parameters['state']) == {'Normal', 'Pico', 'Disrupcion'}, 'Faltan parámetros de algún estado.'
assert set(nominal_plan['policy']) == {'Nominal'}
assert set(buffer_plan['policy']) == {'Comparador +5% demanda'}

print(f'Datos cargados desde: {DATA}')
print(f'Planes puente verificados. Salidas: {OUTPUTS}')
print('Parámetros por estado cargados:')
display(state_parameters)

SHA-256 part1_solution.csv: 79c91cb78ceb0ebed1c98c12bade96d14b2abecf5ecde2b2640df0592476b5ab
SHA-256 part1_solution_buffer.csv: ce6330cb655529ef58c9b292da49789d2c1598e77f589f794f1b55b445192e23
Datos cargados desde: C:\Users\disar\OneDrive\Escritorio\facultad\3_año\segundo_cuat\Optimizacion\Final-OPIO\Datos
Planes puente verificados. Salidas: C:\Users\disar\OneDrive\Escritorio\facultad\3_año\segundo_cuat\Optimizacion\Final-OPIO\outputs
Parámetros por estado cargados:


,state,demand_multiplier,capacity_multiplier,route_availability
0,Normal,1.0,1.00,0.98
1,Pico,1.2,0.85,0.90
2,Disrupcion,1.0,0.60,0.70


## 3. Cadena de Markov (H4)

Cuente transiciones antes de normalizar. Si una fila tiene cero salidas, detenga
la ejecución y comuníquelo: no complete con ceros ni invente autotransiciones.
Valide no negatividad y suma uno por fila; calcule pasos 1, 4, 12 y 26 desde la
condición inicial. Congele la matriz antes de la campaña Monte Carlo.

In [9]:
STATE_NAMES = ['Normal', 'Pico', 'Disrupcion']
TOL = 1e-7

def estimate_transition_matrix(sequence, state_names=STATE_NAMES):
    """Cuenta transiciones observadas y normaliza cada fila de conteos."""
    sequence = list(sequence)
    if len(sequence) < 2:
        raise ValueError('Se necesitan al menos dos observaciones para estimar transiciones.')
    unknown = set(sequence) - set(state_names)
    if unknown:
        raise ValueError(f'Estados no reconocidos: {sorted(unknown)}')

    counts = pd.DataFrame(0, index=state_names, columns=state_names, dtype=int)
    for current, following in zip(sequence[:-1], sequence[1:]):
        counts.loc[current, following] += 1

    outgoing = counts.sum(axis=1)
    if (outgoing == 0).any():
        missing = outgoing.index[outgoing == 0].tolist()
        raise ValueError(f'No hay transiciones salientes observadas para: {missing}')

    matrix = counts.div(outgoing, axis=0)
    assert (matrix.to_numpy() >= 0).all()
    assert np.allclose(matrix.sum(axis=1).to_numpy(), 1.0, atol=TOL)
    return counts, matrix

observations = observations.sort_values('period')
transition_counts, transition_matrix = estimate_transition_matrix(observations['state'])
print('Conteos de transición:')
display(transition_counts)
print('Matriz de transición:')
display(transition_matrix)

# Z_1 es Disrupcion; la semana t usa la distribución inicial multiplicada por P^(t-1).
initial_state = np.zeros(len(STATE_NAMES))
initial_state[STATE_NAMES.index('Disrupcion')] = 1.0
state_distributions = []
for week in (1, 4, 12, 26):
    distribution = initial_state @ np.linalg.matrix_power(transition_matrix.to_numpy(), week - 1)
    state_distributions.append({'semana': week, **dict(zip(STATE_NAMES, distribution))})
display(pd.DataFrame(state_distributions).set_index('semana').round(4))

transition_matrix.to_csv(OUTPUTS / 'markov_matrix.csv', index_label='from_state')

Conteos de transición:


,Normal,Pico,Disrupcion
Normal,66,20,3
Pico,16,37,5
Disrupcion,6,1,5


Matriz de transición:


,Normal,Pico,Disrupcion
Normal,0.741573,0.224719,0.033708
Pico,0.275862,0.637931,0.086207
Disrupcion,0.500000,0.083333,0.416667


,Normal,Pico,Disrupcion
semana,,,
1,0.0000,0.0000,1.0000
4,0.6006,0.2795,0.1199
12,0.5523,0.3623,0.0854
26,0.5522,0.3624,0.0855


## 4. Orden de una semana simulada

Para cada política \(k\):

1. observar/sortear \(Z_t\);
2. generar demanda \(D_{jt}\) y disponibilidad \(A_{ijt}\);
3. ejecutar \(q^k_{ijt}=\min\{\bar x^k_{ij},
   A_{ijt}m^{\mathrm{cap}}_{Z_t}u_{ij}\}\);
4. sumar recepciones, atender demanda, registrar venta perdida;
5. conservar hasta \(K_j\), descartar exceso y acumular costos.

No hay pedidos pendientes. La demanda no atendida se pierde en esa semana.
Ambas políticas usan el mismo futuro en cada repetición.

## 5. Funciones puras y pruebas

generate_environment crea un futuro completo sin conocer la política.
evaluate_policy recibe ese futuro y un plan fijo. Pruebe al menos: límite
determinista, capacidad cero, demanda cero, inventario acotado, balance semanal,
no negatividad y repetición exacta con la misma semilla.

In [10]:
def generate_environment(
    rng,
    transition_matrix,
    destinations,
    routes,
    state_parameters,
    horizon=26,
):
    """Genera el futuro completo. Demanda Poisson y disponibilidad Bernoulli, independientes condicionalmente al estado."""
    if horizon < 1:
        raise ValueError('horizon debe ser al menos 1.')

    params = state_parameters.set_index('state').loc[STATE_NAMES]
    state_path = np.empty(horizon, dtype=object)
    state_path[0] = 'Disrupcion'
    for week in range(1, horizon):
        probs = transition_matrix.loc[state_path[week - 1], STATE_NAMES].to_numpy(dtype=float)
        state_path[week] = rng.choice(STATE_NAMES, p=probs)

    demand_multiplier = np.array([params.loc[z, 'demand_multiplier'] for z in state_path], dtype=float)
    demand_rate = demand_multiplier[:, None] * destinations['nominal_demand'].to_numpy(dtype=float)[None, :]
    demand = rng.poisson(demand_rate)

    availability_probability = np.array(
        [params.loc[z, 'route_availability'] for z in state_path], dtype=float
    )
    route_available = rng.random((horizon, len(routes))) < availability_probability[:, None]
    capacity_multiplier = np.array(
        [params.loc[z, 'capacity_multiplier'] for z in state_path], dtype=float
    )

    return {
        'states': state_path,
        'demand': demand,
        'route_available': route_available,
        'capacity_multiplier': capacity_multiplier,
        'destinations': destinations.copy().reset_index(drop=True),
        'routes': routes.copy().reset_index(drop=True),
    }


def evaluate_policy(plan, environment):
    """Evalúa un plan fijo en un futuro; no reoptimiza ni genera azar."""
    routes = environment['routes']
    destinations = environment['destinations']
    route_ids = routes['route_id'].tolist()
    destination_names = destinations['destination'].tolist()
    plan_by_route = plan.set_index('route_id').loc[route_ids]
    planned_flow = plan_by_route['planned_flow'].to_numpy(dtype=float)

    route_destination = np.array([
        destination_names.index(name) for name in routes['destination']
    ])
    route_cost = routes['unit_cost'].to_numpy(dtype=float)
    route_capacity = routes['capacity'].to_numpy(dtype=float)
    demand_penalty = destinations['shortage_penalty'].to_numpy(dtype=float)
    storage_capacity = destinations['storage_capacity'].to_numpy(dtype=float)
    holding_rate = destinations['holding_cost'].to_numpy(dtype=float)
    waste_rate = destinations['waste_cost'].to_numpy(dtype=float)
    inventory = destinations['initial_inventory'].to_numpy(dtype=float).copy()

    total_demand = total_served = shortage_units = waste_units = 0.0
    transport_cost = shortage_cost = holding_cost = waste_cost = 0.0
    shortage_weeks = 0
    inventory_sum = 0.0

    for week in range(len(environment['states'])):
        actual_flow = np.minimum(
            planned_flow,
            environment['route_available'][week]
            * environment['capacity_multiplier'][week]
            * route_capacity,
        )
        receipts = np.zeros(len(destination_names), dtype=float)
        np.add.at(receipts, route_destination, actual_flow)

        demand = environment['demand'][week].astype(float)
        opening_inventory = inventory.copy()
        available_stock = opening_inventory + receipts
        served = np.minimum(available_stock, demand)
        lost = demand - served
        excess = np.maximum(available_stock - demand, 0.0)
        discarded = np.maximum(excess - storage_capacity, 0.0)
        inventory = np.minimum(excess, storage_capacity)

        # Balances independientes de producto y demanda para cada semana y zona.
        assert np.all(np.abs(opening_inventory + receipts - served - inventory - discarded) <= TOL)
        assert np.all(np.abs(served + lost - demand) <= TOL)
        assert (actual_flow >= -TOL).all() and (lost >= -TOL).all()
        assert (inventory >= -TOL).all() and (inventory <= storage_capacity + TOL).all()

        total_demand += demand.sum()
        total_served += served.sum()
        shortage_units += lost.sum()
        waste_units += discarded.sum()
        shortage_weeks += int(np.any(lost > TOL))
        inventory_sum += inventory.sum()
        transport_cost += float(np.dot(actual_flow, route_cost))
        shortage_cost += float(np.dot(lost, demand_penalty))
        holding_cost += float(np.dot(inventory, holding_rate))
        waste_cost += float(np.dot(discarded, waste_rate))

    horizon = len(environment['states'])
    service_level = total_served / total_demand if total_demand > 0 else 1.0
    return {
        'total_cost': transport_cost + shortage_cost + holding_cost + waste_cost,
        'transport_cost': transport_cost,
        'shortage_cost': shortage_cost,
        'holding_cost': holding_cost,
        'waste_cost': waste_cost,
        'service_level': service_level,
        'any_shortage': float(shortage_units > TOL),
        'fraction_weeks_with_shortage': shortage_weeks / horizon,
        'shortage_units': shortage_units,
        'average_inventory': inventory_sum / horizon,
        'waste_units': waste_units,
    }

## 6. Piloto de 200 (H4)

El piloto sirve para detectar errores, ocupación de estados y escala; no para
elegir retrospectivamente métricas. Registre si mantiene o corrige el protocolo.
Si lo corrige, repita el piloto antes de autorizar la campaña.

In [11]:
import copy

MASTER_SEED = 20260816
PILOT_REPLICATIONS = 200
FINAL_REPLICATIONS = 1200
HORIZON = 26

# Una sola secuencia para la campaña: el piloto usa exactamente sus primeras 200 semillas.
replication_seeds = np.random.SeedSequence(MASTER_SEED).spawn(FINAL_REPLICATIONS)
POLICIES = {'Nominal': nominal_plan, 'Resguardo': buffer_plan}

def run_campaign(seed_sequences):
    """Simula una trayectoria por réplica y evalúa ambas políticas en ese mismo futuro."""
    rows = []
    state_counts = pd.Series(0, index=STATE_NAMES, dtype=int)
    for replication, child_seed in enumerate(seed_sequences):
        environment = generate_environment(
            rng=np.random.default_rng(child_seed),
            transition_matrix=transition_matrix,
            destinations=destinations,
            routes=routes,
            state_parameters=state_parameters,
            horizon=HORIZON,
        )
        state_counts += pd.Series(environment['states']).value_counts().reindex(STATE_NAMES, fill_value=0)
        for policy_name, plan in POLICIES.items():
            rows.append({
                'replication': replication,
                'policy': policy_name,
                **evaluate_policy(plan, environment),
            })
    results = pd.DataFrame(rows)
    state_share = state_counts / (len(seed_sequences) * HORIZON)
    return results, state_share

def run_invariant_checks():
    """Chequeos pequeños del simulador antes de iniciar el piloto."""
    seed = 314159
    first = generate_environment(
        np.random.default_rng(seed), transition_matrix, destinations, routes,
        state_parameters, horizon=HORIZON,
    )
    repeated = generate_environment(
        np.random.default_rng(seed), transition_matrix, destinations, routes,
        state_parameters, horizon=HORIZON,
    )
    assert np.array_equal(first['states'], repeated['states'])
    assert np.array_equal(first['demand'], repeated['demand'])
    assert np.array_equal(first['route_available'], repeated['route_available'])

    fixed = copy.deepcopy(first)
    fixed['states'][:] = 'Normal'
    fixed['demand'][:] = 0
    fixed['route_available'][:] = True
    fixed['capacity_multiplier'][:] = 1.0
    result = evaluate_policy(nominal_plan, fixed)
    expected_transport = HORIZON * float(
        np.dot(nominal_plan.set_index('route_id').loc[routes['route_id'], 'planned_flow'], routes['unit_cost'])
    )
    assert abs(result['transport_cost'] - expected_transport) <= TOL
    assert result['shortage_units'] == 0 and result['service_level'] == 1.0

    no_routes = copy.deepcopy(fixed)
    no_routes['route_available'][:] = False
    no_capacity_result = evaluate_policy(nominal_plan, no_routes)
    assert no_capacity_result['transport_cost'] == 0.0

    first_result = evaluate_policy(nominal_plan, first)
    repeated_result = evaluate_policy(nominal_plan, repeated)
    assert first_result == repeated_result

run_invariant_checks()

pilot_results, pilot_state_share = run_campaign(replication_seeds[:PILOT_REPLICATIONS])
print('Ocupación de estados en el piloto:')
display(pilot_state_share.rename('fracción').to_frame())
print('Diagnóstico del piloto (promedios):')
display(
    pilot_results.groupby('policy')[
        ['total_cost', 'service_level', 'shortage_units', 'average_inventory', 'waste_units']
    ].mean().round(3)
)
assert len(pilot_results) == PILOT_REPLICATIONS * len(POLICIES)
assert len(replication_seeds) == FINAL_REPLICATIONS

Ocupación de estados en el piloto:


,fracción
Normal,0.531731
Pico,0.325192
Disrupcion,0.143077


Diagnóstico del piloto (promedios):


,total_cost,service_level,shortage_units,average_inventory,waste_units
policy,,,,,
Nominal,100499.196,0.804,1151.018,16.647,0.115
Resguardo,83223.640,0.850,880.754,35.994,3.366


## 7. Campaña final emparejada

Genere una sola trayectoria por repetición y pásela a ambas políticas. Informe
media, error estándar e intervalo del 95 % de la diferencia
resguardo - nominal para costo y nivel de servicio. Incluya probabilidad de
algún faltante, fracción de semanas con faltante, unidades faltantes, inventario
y descarte.

In [12]:
# Campaña final emparejada: una trayectoria común por réplica para ambas políticas.
final_results, final_state_share = run_campaign(replication_seeds)

metrics = [
    'total_cost', 'transport_cost', 'shortage_cost', 'holding_cost', 'waste_cost',
    'service_level', 'any_shortage', 'fraction_weeks_with_shortage',
    'shortage_units', 'average_inventory', 'waste_units',
]
summary_rows = []
for policy_name in POLICIES:
    policy_results = final_results.loc[final_results['policy'] == policy_name]
    for metric in metrics:
        values = policy_results[metric].to_numpy(dtype=float)
        summary_rows.append({
            'policy': policy_name,
            'metric': metric,
            'mean': values.mean(),
            'standard_error': values.std(ddof=1) / np.sqrt(len(values)),
        })
montecarlo_summary = pd.DataFrame(summary_rows)

# Diferencia emparejada con convención Resguardo - Nominal.
paired_rows = []
for metric in ('total_cost', 'service_level'):
    paired = final_results.pivot(index='replication', columns='policy', values=metric)
    difference = paired['Resguardo'] - paired['Nominal']
    n = len(difference)
    mean_difference = difference.mean()
    standard_error = difference.std(ddof=1) / np.sqrt(n)
    critical_value = student_t.ppf(0.975, df=n - 1)
    paired_rows.append({
        'metric': metric,
        'contrast': 'Resguardo - Nominal',
        'mean_difference': mean_difference,
        'standard_error': standard_error,
        'ci95_low': mean_difference - critical_value * standard_error,
        'ci95_high': mean_difference + critical_value * standard_error,
        'replications': n,
    })
paired_comparison = pd.DataFrame(paired_rows)

transition_matrix.to_csv(OUTPUTS / 'markov_matrix.csv', index_label='from_state')
montecarlo_summary.to_csv(OUTPUTS / 'montecarlo_summary.csv', index=False)
paired_comparison.to_csv(OUTPUTS / 'paired_comparison.csv', index=False)

print('Resumen Monte Carlo (medias y errores estándar):')
display(montecarlo_summary.round(4))
print('Comparación emparejada, Resguardo - Nominal:')
display(paired_comparison.round(5))
print('Ocupación de estados en campaña final:')
display(final_state_share.rename('fracción').to_frame())

Resumen Monte Carlo (medias y errores estándar):


,policy,metric,mean,standard_error
0,Nominal,total_cost,100063.0969,640.4441
1,Nominal,transport_cost,18541.4784,26.1635
2,Nominal,shortage_cost,81370.1921,664.9324
3,Nominal,holding_cost,151.3456,2.5402
4,Nominal,waste_cost,0.0809,0.0240
5,Nominal,service_level,0.8054,0.0015
6,Nominal,any_shortage,1.0000,0.0000
7,Nominal,fraction_weeks_with_shortage,0.9221,0.0021
8,Nominal,shortage_units,1143.3324,9.3231
9,Nominal,average_inventory,16.5995,0.2777


Comparación emparejada, Resguardo - Nominal:


,metric,contrast,mean_difference,standard_error,ci95_low,ci95_high,replications
0,total_cost,Resguardo - Nominal,-17486.22987,68.53019,-17620.68230,-17351.77745,1200
1,service_level,Resguardo - Nominal,0.04671,0.00015,0.04642,0.04701,1200


Ocupación de estados en campaña final:


,fracción
Normal,0.531378
Pico,0.328814
Disrupcion,0.139808


## 8. Recomendación y cierre

La recomendación debe citar estimando, intervalo, magnitud práctica, objetivo de
servicio, supuestos y límite de validez. No llame “óptimo estocástico” a un plan
que sólo fue optimizado con datos nominales.

### H5 — 03/11

- [ ] Otra persona ejecutó el estado congelado de H4 en un entorno limpio.
- [ ] El informe completo y la presentación preliminares coinciden con las salidas.
- [ ] Se registraron defectos y correcciones priorizadas.
- [ ] No se incorporó análisis nuevo después del segundo parcial.

### Entrega — 06/11

- [ ] Restart and Run All funciona con rutas relativas.
- [ ] Salidas e informe coinciden y tienen hashes/versiones.
- [ ] Registro de IA: aceptar/modificar/rechazar + prueba.
- [ ] Cada integrante puede defender modelo, código, evidencia y límites.